In [124]:
# pip install -U scikit-learn

In [125]:
import pandas as pd
import numpy as np

# Giả sử 'df_raw' là dataframe chứa 31 thuộc tính ban đầu của bạn
# Load dataset

DATA_PATH = "../../data/processed/phone_specs_preprocessed_v2.csv"

df = pd.read_csv(DATA_PATH)
display(df.isna().sum())

product_name                0
brand                       0
url                         0
price_vnd                   0
price_segment               0
promotion                  19
phone_type                  0
os_family                   0
os_version                 74
chip_name                   0
chip_brand                  0
screen_size_inch            1
panel                       0
panel_group                 0
refresh_rate_hz            28
resolution_width           27
resolution_height          27
resolution_total_pixels    27
ram_gb                     28
storage_gb                  0
battery_mah                52
fast_charge_w              36
front_camera_mp             3
rear_main_camera_mp         7
rear_camera_count           7
max_video_resolution        0
fps_at_max_resolution       0
video_resolution_rank       0
weight_g                   54
ip_status                   0
dtype: int64

## Rating màn hình

In [126]:
df['refresh_rate_hz_fixed'] = df['refresh_rate_hz'].fillna(60)

In [127]:
df['screen_size_fixed'] = df.groupby('price_segment')['screen_size_inch'].transform(lambda x: x.fillna(x.median()))
df['resolution_total_pixels'] = df.groupby('price_segment')['resolution_total_pixels'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
)

features = ['screen_size_fixed', 'panel_group', 'resolution_total_pixels']
display(df[features].isna().sum())

screen_size_fixed          0
panel_group                0
resolution_total_pixels    0
dtype: int64

In [128]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Giả lập tập dữ liệu lớn hơn một chút để K-Means hoạt động ổn định
# data = {
#     'screen_size': [15.6, 14.0, 27.0, 13.3, 15.6, 11.6, 32.0, 14.0],
#     'panel_group': ['TN', 'IPS', 'IPS', 'OLED', 'VA', 'TN', 'Mini-LED', 'OLED'],
#     'resolution_total_pixels': [2073600, 2073600, 8294400, 5184000, 2073600, 1049088, 8294400, 3686400]
# }
# df = pd.DataFrame(data)

# 2. Xử lý thuộc tính chữ (Ordinal Encoding dựa trên thứ tự chất lượng)
panel_map = {'Unknown':0, 'LCD': 2, 'AMOLED': 3}
df['panel_score'] = df['panel_group'].map(panel_map)

#Chọn các tính năng để đưa vào K-Means
features = ['screen_size_fixed', 'panel_score', 'resolution_total_pixels']
X = df[features]

# 3. CHUẨN HÓA DỮ LIỆU (Bắt buộc phải có với K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. CHẠY K-MEANS (Chia làm 4 cụm tương ứng 4 Tiers)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster_raw'] = kmeans.fit_predict(X_scaled)

# 5. ĐỊNH DANH LẠI THỨ TỰ (Sắp xếp cụm từ thấp đến cao theo độ phân giải trung bình)
# Tính độ phân giải trung bình của từng cụm thô
cluster_order = df.groupby('cluster_raw')['resolution_total_pixels'].mean().sort_values().index

# Tạo bản đồ ánh xạ từ cụm ngẫu nhiên sang tên Tier chuẩn
tier_names = [1,2,3,4]
cluster_mapping = {raw_cluster: tier_names[i] for i, raw_cluster in enumerate(cluster_order)}

# Gán nhãn cuối cùng
df['display_tier'] = df['cluster_raw'].map(cluster_mapping)

# Xem kết quả kiểm chứng
print(df[df['price_segment']=='flagship'][['screen_size_inch', 'panel_group', 'resolution_height', 'display_tier']])

     screen_size_inch panel_group  resolution_height  display_tier
3                6.20      AMOLED             3200.0             2
9                8.12      AMOLED                NaN             4
11               6.30      AMOLED             2340.0             2
17               6.90      AMOLED             3200.0             3
19               6.10      AMOLED             2532.0             2
..                ...         ...                ...           ...
374              6.60      AMOLED             2340.0             3
378              8.00      AMOLED             2184.0             4
381              7.60      AMOLED             2176.0             4
384              6.70      AMOLED             2778.0             3
386              7.60      AMOLED             2176.0             4

[111 rows x 4 columns]


## Rating chip

In [129]:
print(df['chip_name'].head())

0           Helio G85 (12nm)
1         Snapdragon 6 Gen 3
2                 Exynos 850
3         Exynos 990 (7 nm+)
4    Qualcomm Snapdragon 685
Name: chip_name, dtype: str


In [130]:
import re
path_chip_rating = '../../data/raw/processor/processors_ratings.csv'
df_chip_rating = pd.read_csv(path_chip_rating)
# ---------------------------------------------------------
# 1. BƯỚC CHUẨN BỊ (Làm sạch cơ bản)
# ---------------------------------------------------------
# Chúng ta vẫn cần một chút làm sạch nhẹ để xóa các ký tự cản trở như ®, ™ và đưa về chữ thường
def light_clean(text):
    if pd.isna(text): 
        return ""
    text = str(text).lower()
    text = text.replace('(', '').replace(')', '')
    text = text.replace('+','')
    
    # 2. Xóa các ký tự thương hiệu như ®, ™
    text = re.sub(r'[®™]', '', text) 

    text = text.replace('thế hệ thứ', 'gen')
    text = text.replace('thế hệ', 'gen')

    
    
    # 3. Dọn dẹp khoảng trắng thừa (biến nhiều khoảng trắng liên tiếp thành 1 khoảng trắng)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def normalize_aliases(text):
    if '8 elite' in text:
        if 'gen 5' in text:
            return 'snapdragon 8 elite gen 5'
        else:
            # Ép tất cả các bản 8 elite còn lại (bản trơn, 3nm, for galaxy, gen 4...) về đúng tên chuẩn
            return 'snapdragon 8 elite gen 4'
    if 'apple a15' in text or text == 'a15': return 'a15 bionic'
    if 'chip a18' in text or text == 'a18': return 'apple a18'
    if 't612' in text: return 'tiger t612'
    if 't616' in text: return 'tiger t616'
    if 't606' in text: return 'tiger t606'
    if 'g99 ultimate' in text: return 'helio g99 ultimate'
    if 'g99' in text and 'helio' not in text: return 'helio g99'
    if 'g100' in text and 'helio' not in text: return 'helio g100'
    if '7025' in text and 'ultra' in text: return 'dimensity 7025 ultra'
    if '8895' in text: return 'exynos 8895'
    return text

# Áp dụng cho DataFrame chính
df['chip_clean'] = df['chip_name'].apply(light_clean).apply(normalize_aliases)

# Chuẩn hóa cột file CSV
df_chip_rating.columns = df_chip_rating.columns.str.lower()
df_chip_rating['processor_clean'] = df_chip_rating['processor'].apply(light_clean)
df_chip_rating['brand_clean'] = df_chip_rating['brand'].astype(str).str.lower().str.strip()
# ---------------------------------------------------------
# 2. TẠO TỪ ĐIỂN VÀ TÍNH ĐIỂM MIN TỰ ĐỘNG TỪ CỘT BRAND
# ---------------------------------------------------------
# Từ điển Chip
chip_df_grouped = df_chip_rating.groupby('processor_clean')['rating'].max().reset_index()
chip_dict = dict(zip(chip_df_grouped['processor_clean'], chip_df_grouped['rating']))
sorted_chip_names = sorted(chip_dict.keys(), key=len, reverse=True)

# Tự động tạo Dictionary lấy điểm Min của cả 8 hãng có trong file CSV
# Kết quả sẽ có dạng: {'qualcomm': 24, 'apple': 24, 'mediatek': 23, 'samsung': 23, ...}
brand_mins = df_chip_rating.groupby('brand_clean')['rating'].min().to_dict()

# ---------------------------------------------------------
# 3. QUÉT CHUỖI VÀ GÁN ĐIỂM (CẬP NHẬT 8 HÃNG)
# ---------------------------------------------------------
def get_final_chip_score(messy_name):
    if not messy_name or messy_name == 'unknown':
        return 0
        
    # 3.1: Dò khớp tên chip chính xác
    for clean_name in sorted_chip_names:
        if clean_name in messy_name:
            return chip_dict[clean_name]
            
    # 3.2: Dò từ khóa (Keywords) để nhận diện 8 hãng và giáng cấp xuống Min score
    if any(k in messy_name for k in ['qualcomm', 'snapdragon', 'sdm']): 
        return brand_mins.get('qualcomm', 0)
        
    if any(k in messy_name for k in ['apple', 'bionic', 'chip a']): 
        return brand_mins.get('apple', 0)
        
    if any(k in messy_name for k in ['mediatek', 'dimensity', 'helio', 'mt6', 'mt8']): 
        return brand_mins.get('mediatek', 0)
        
    if any(k in messy_name for k in ['samsung', 'exynos']): 
        return brand_mins.get('samsung', 0)
        
    if any(k in messy_name for k in ['xiaomi', 'surge', 'xring']): 
        return brand_mins.get('xiaomi', 0)
        
    if any(k in messy_name for k in ['hisilicon', 'kirin']): 
        return brand_mins.get('hisilicon', 0)
        
    if any(k in messy_name for k in ['google', 'tensor']): 
        return brand_mins.get('google', 0)
        
    if any(k in messy_name for k in ['unisoc', 'tiger', 'spreadtrum', 'ums', ' t6', ' t7']): 
        return brand_mins.get('unisoc', 0)
    
    # 3.3: Dữ liệu rác hoàn toàn (Ví dụ: "8 nhân", "Mali-G57")
    return 0

# ---------------------------------------------------------
# 4. THỰC THI VÀ DỌN DẸP
# ---------------------------------------------------------
df['chip_rating'] = df['chip_clean'].apply(get_final_chip_score)
df = df.drop(columns=['chip_clean'])

# Kiểm tra dữ liệu rớt đài
zero_score_chips = df[df['chip_rating'] == 0]['chip_name'].unique()
print("Dữ liệu rác (0 điểm):", zero_score_chips)

Dữ liệu rác (0 điểm): <StringArray>
[                                    '8 nhân',
 '10 nhân, 3.3 GHz, 2.74GHz, 2.36GHz, 1.8GHz',
                                      'T7200',
                                   'Mali-G57',
                                    'Unknown']
Length: 5, dtype: str


# Camera scoring

In [131]:
def print_missing_percent(df):
    missing_count = df.isnull().sum()

    # Tính tỷ lệ phần trăm giá trị thiếu ở từng cột
    missing_percentage = (df.isnull().sum() / len(df)) * 100

    # Gộp thành một DataFrame thống kê cho trực quan
    missing_summary = pd.DataFrame({
        'Số lượng thiếu': missing_count,
        'Tỷ lệ phần trăm (%)': missing_percentage
    })
    print(missing_summary)
rating_path = "../../data/raw/rating/phone_ratings.csv" 
rating_df = pd.read_csv(rating_path)
print("--- THỐNG KÊ GIÁ TRỊ THIẾU ---")
print_missing_percent(rating_df)
print("\n" + "="*50 + "\n")


# =====================================================================
# BƯỚC 2: DROP CÁC HÀNG KHÔNG CÓ CẢ CAMERA_RATING VÀ DISPLAY_RATING
# =====================================================================

print("--- XỬ LÝ DROP DỮ LIỆU THIẾU ---")
print(f"Số lượng dòng ban đầu: {len(rating_df)}")

# subset: chỉ định các cột cần kiểm tra
# how='all': xóa nếu CẢ 2 cột đều không có giá trị (NaN)
# inplace=False (mặc định) sẽ trả về một DataFrame mới, bạn có thể gán lại vào rating_df nếu muốn ghi đè
rating_df_cleaned = rating_df.dropna(subset=['camera_rating', 'display_rating'], how='all')

print(f"Số lượng dòng sau khi drop: {len(rating_df_cleaned)}")

# Hiển thị dữ liệu sau khi đã làm sạch
print("\nDữ liệu sau khi làm sạch:")
missing_p = (rating_df_cleaned.isnull().sum() / len(rating_df_cleaned)) * 100
print_missing_percent(rating_df_cleaned)

--- THỐNG KÊ GIÁ TRỊ THIẾU ---
                Số lượng thiếu  Tỷ lệ phần trăm (%)
product_name                 0             0.000000
camera_rating              154            37.378641
display_rating             237            57.524272


--- XỬ LÝ DROP DỮ LIỆU THIẾU ---
Số lượng dòng ban đầu: 412
Số lượng dòng sau khi drop: 286

Dữ liệu sau khi làm sạch:
                Số lượng thiếu  Tỷ lệ phần trăm (%)
product_name                 0             0.000000
camera_rating               28             9.790210
display_rating             111            38.811189


In [132]:
import re
from rapidfuzz import process, fuzz

# 1. Hàm tiền xử lý để xóa cấu hình nhiễu (RAM, bộ nhớ, 4G/5G, VN/A)
def clean_phone_name(name):
    if pd.isna(name):
        return ""
    name = str(name)
    # Chuyển về chữ thường
    name = name.lower()
    # Xóa thông số cấu hình gây nhiễu (RAM, ROM, 4G, 5G)
    name = re.sub(r'\b\d+gb\b', '', name)
    #name = re.sub(r'\b[45]g\b', '', name)
    # Xóa các cụm dung lượng như 8gb, 128gb, 12gb, 512gb...
    name = re.sub(r'\b\d+gb\b', '', name)
    # Xóa chữ 4g, 5g
    name = re.sub(r'\b[45]g\b', '', name)
    # Xóa các ký tự đặc biệt và phần đuôi rác như | chính hãng vn/a
    name = re.sub(r'\|.*$', '', name) 
    name = name.replace('chính hãng vn/a', '')
    # Xóa khoảng trắng thừa
    name = ' '.join(name.split())
    return name

# 2. Hàm match tên cải tiến
def find_best_match_rapidfuzz(name, choices, threshold=75):
    cleaned_name = clean_phone_name(name)
    if not cleaned_name:
        return None
    
    # Tạo danh sách choices đã clean để so sánh
    cleaned_choices = {clean_phone_name(c): c for c in choices}
    
    # Rapidfuzz sử dụng scorer trực tiếp từ thư viện
    # extractOne của rapidfuzz trả về: (matched_string, score, index)
    result = process.extractOne(cleaned_name, list(cleaned_choices.keys()), scorer=fuzz.token_sort_ratio)
    
    if result is None:
        return None
        
    matched_clean_key, score, _ = result
    actual_match_name = cleaned_choices[matched_clean_key]
    
    # BỘ LỌC CHẶT CHẼ: Ép phải trùng mã số dòng máy (ví dụ: 11, 13, 21, 36)
    model_codes = re.findall(r'\b\d+[a-z]?\b|\b[a-z]\d+\b', cleaned_name)
    if model_codes:
        if not any(code in clean_phone_name(actual_match_name) for code in model_codes):
            return None # Trả về None nếu lệch số dòng máy (như A36 biến thành S8)

    if score >= threshold:
        return actual_match_name
    return None

# --- CHẠY THỬ NGHIỆM VỚI DỮ LIỆU CỦA BẠN ---
# Áp dụng hàm match mới

# Tạo danh sách các tên điện thoại trong file rating để đối chiếu
rating_choices = rating_df['product_name'].tolist()
rating_choices = [re.sub(r'^apple\s+', '', choice, flags=re.IGNORECASE) for choice in rating_choices]
df['matched_name'] = df['product_name'].apply(lambda x: find_best_match_rapidfuzz(x, rating_choices, threshold=75))

# Merge kết quả để lấy cột camera_rating sang tập df cần đánh giá
df_final = pd.merge(df, rating_df, left_on='matched_name', right_on='product_name', how='left', suffixes=('', '_rating_file'))
# Bỏ cột thừa sau khi merge
df_final = df_final.drop(columns=['product_name_rating_file'])

print("--- KẾT QUẢ SẦU KHI MATCH NÊN ---")
print(df_final[['product_name', 'matched_name']].head(10))



--- KẾT QUẢ SẦU KHI MATCH NÊN ---
                            product_name                 matched_name
0                     vivo Y28 8GB 128GB                          NaN
1        Samsung Galaxy A36 5G 8GB 128GB                          NaN
2                Samsung Galaxy A21s 6GB       Samsung Galaxy A52s 5G
3                     Samsung Galaxy S20          Samsung Galaxy S20+
4                      OPPO A6t 4GB 64GB                 Oppo A16s 5G
5            ASUS ROG Phone 6 16GB 512GB             Asus ROG Phone 6
6                Nubia V80 Max 8GB 128GB                          NaN
7                Xiaomi 15 5G 12GB 512GB                    Xiaomi 15
8  Xiaomi Redmi Note 13 Pro 4G 8GB 128GB  Xiaomi Redmi Note 13 Pro 5G
9                OPPO Find N5 16GB 512GB                 Oppo Find N5


In [133]:
# Đếm số lượng dòng bị thiếu (NaN/None) trong cột matched_name
missing_match_count = df_final['matched_name'].isnull().sum()

# Tính tỷ lệ phần trăm bị thiếu
missing_match_percentage = (missing_match_count / len(df_final)) * 100

print(f"Số lượng điện thoại KHÔNG match được: {missing_match_count} dòng")
print(f"Tỷ lệ không match được: {missing_match_percentage:.2f}%")

# (Tùy chọn) In ra danh sách các sản phẩm không match được để bạn tiện kiểm tra lý do
print("\nDanh sách các dòng máy không match được:")
no_match_df = df_final[df_final['matched_name'].isnull()]
print(no_match_df['product_name'])

Số lượng điện thoại KHÔNG match được: 170 dòng
Tỷ lệ không match được: 43.93%

Danh sách các dòng máy không match được:
0                   vivo Y28 8GB 128GB
1      Samsung Galaxy A36 5G 8GB 128GB
6              Nubia V80 Max 8GB 128GB
10         Samsung Galaxy S20 FE 256GB
11       Samsung Galaxy S26 12GB 512GB
                    ...               
375    Samsung Galaxy A17 5G 8GB 128GB
376        HONOR 400 Pro 5G 12GB 512GB
377       Samsung Galaxy A07 4GB 128GB
382                 OPPO A58 8GB 128GB
383       Samsung Galaxy A06 4GB 128GB
Name: product_name, Length: 170, dtype: str


In [134]:
def get_brand(name):
    if pd.isna(name): return "unknown"
    name = str(name).lower()
    for brand in ['samsung', 'apple', 'iphone', 'xiaomi', 'oppo', 'vivo', 'realme', 'asus', 'nubia', 'honor', 'huawei']:
        if brand in name:
            return 'apple' if brand == 'iphone' else brand
    return "other"

rating_df['brand'] = rating_df['product_name'].apply(get_brand)
brand_medians = rating_df.groupby('brand')['camera_rating'].median()
global_median = rating_df['camera_rating'].median()

# Điền các hãng không có trong file rating bằng điểm trung vị tổng
brand_medians = brand_medians.fillna(global_median)
def smart_impute_camera(row, rating_choices, rating_df):
    p_name = row['product_name']
    matched = row['matched_name']
    brand = row['brand'].lower()
    
    # TH 1: Đã match thành công ở bước trước và CÓ điểm camera
    if pd.notna(matched):
        # Lấy điểm từ file rating sang
        score_series = rating_df[rating_df['product_name'] == matched]['camera_rating']
        if not score_series.empty and pd.notna(score_series.values[0]):
            return score_series.values[0]
            
        # TH 2: Match thành công tên nhưng dòng đó trong file rating BỊ TRỐNG điểm camera
        # Giải pháp: Lấy điểm trung vị của thương hiệu đó
        return brand_medians.get(brand, global_median)

    # TH 3: KHÔNG match được tên (nằm trong nhóm 175 dòng bị thiếu)
    # Thử cứu bằng cách hạ threshold xuống 60 và chỉ so sánh độ tương đồng chuỗi đơn thuần
    cleaned_name = clean_phone_name(p_name) # Gọi lại hàm clean ở bài trước
    cleaned_choices = {clean_phone_name(c): c for c in rating_choices}
    
    result = process.extractOne(cleaned_name, list(cleaned_choices.keys()), scorer=fuzz.token_set_ratio)
    if result:
        matched_clean_key, score, _ = result
        # Nếu hạ tiêu chuẩn xuống 60 mà khớp, chấp nhận kết quả cứu vớt này
        if score >= 60:
            actual_match = cleaned_choices[matched_clean_key]
            score_series = rating_df[rating_df['product_name'] == actual_match]['camera_rating']
            if not score_series.empty and pd.notna(score_series.values[0]):
                return score_series.values[0]

    # TH 4: Hoàn toàn bó tay (không trùng hãng, không trùng tên - ví dụ Masstel)
    # Giải pháp: Gán điểm theo trung vị của hãng đó, hoặc trung vị toàn tập dữ liệu
    return brand_medians.get(brand, global_median)

# =====================================================================
# THỰC THI CỨU DỮ LIỆU
# =====================================================================
# Áp dụng hàm điền điểm thông minh cho toàn bộ tập dữ liệu cần đánh giá
df_final['camera_rating_final'] = df_final.apply(
    lambda r: smart_impute_camera(r, rating_choices, rating_df), axis=1
)

# Kiểm tra lại xem còn dòng nào bị trống điểm camera nữa không
print(df_final['camera_rating_final'].isna().sum())

0


## KBRS scoring

In [135]:
df_final['ip_status'] = df_final['ip_status'].astype(str).str.lower().str.strip()

df_final['ip_status'] = np.where(df_final['ip_status'] == 'certified', 1, 0)

print(df_final['ip_status'].unique())

[0]


In [136]:
# Bọc lót dữ liệu thiếu (NaN): Điền bằng số trung vị (50%) là 194.0g
df_final['weight_g'] = df_final['weight_g'].fillna(194.0)

df_final['weight_g'] = df_final['weight_g'].fillna(194.0)

# 2. Định nghĩa các điều kiện mốc cắt cho 2 nhóm đầu (Độ dài = 2)
conditions = [
    df_final['weight_g'] < 185.0,                               # Nhóm Nhẹ
    (df_final['weight_g'] >= 185.0) & (df_final['weight_g'] < 210.0)   # Nhóm Vừa
]

# 3. Định nghĩa nhãn tương ứng cho 2 điều kiện trên (Độ dài phải = 2)
# Điều kiện 1 ứng với nhãn 1, Điều kiện 2 ứng với nhãn 2
tier_labels = [1, 2] 

# 4. Tạo cột phân loại mới trong dataframe
# Những máy không thỏa mãn 2 điều kiện trên tự động nhận giá trị default là 3 (Nhóm Heavy)
df_final['weight_tier'] = np.select(conditions, tier_labels, default=3)

# Kiểm tra số lượng máy được phân bổ vào từng nhóm
print(df_final['weight_tier'].value_counts())

weight_tier
2    238
3     75
1     74
Name: count, dtype: int64


In [137]:
final_export_features = [
    # ---------------------------------------------------------
    # NHÓM 1: THÔNG TIN ĐỊNH DANH & THƯƠNG MẠI (Hiển thị + Lọc segment)
    # ---------------------------------------------------------
    'product_name',       # Tên sản phẩm
    'brand',              # Thương hiệu
    'url',                # Đường dẫn mua hàng
    'price_vnd',          # Giá tiền bằng số để lọc khoảng giá (ví dụ: từ 5tr - 10tr)
    'price_segment',      # Phân khúc giá (budget, mid-range, flagship) để filter cứng
    'promotion',          # Thông tin khuyến mãi
    
    # ---------------------------------------------------------
    # NHÓM 2: BỘ CHỈ SỐ ĐIỂM TOÀN CỤC (Dùng để Rank kết quả + Vẽ Progress Bar trên UI)
    # ---------------------------------------------------------
    'score_perf',         # Điểm hiệu năng tổng hợp
    'score_cam',          # Điểm camera tổng hợp
    'score_batt',         # Điểm pin & sạc tổng hợp
    'score_disp',         # Điểm màn hình tổng hợp
    
    # ---------------------------------------------------------
    # NHÓM 3: CÁC BIẾN ĐIỀU KIỆN CỨNG (Dùng cho Hard Filters trong Ma trận cấu hình)
    # ---------------------------------------------------------
    'weight_tier',        # Phân loại cân nặng (1: Nhẹ, 2: Vừa, 3: Nặng)
    'ip_status',       # Biến nhị phân (1/0) chống nước tốt hay không (Nếu bạn đã tạo ở bước trước)
    
    # ---------------------------------------------------------
    # NHÓM 4: CHI TIẾT PHẦN CỨNG (Hiện bảng thông số kỹ thuật chi tiết cho User đọc)
    # ---------------------------------------------------------
    # Cấu hình chính
    'chip_name', 'ram_gb', 'storage_gb', 'os_family', 'os_version',
    
    # Màn hình
    'screen_size_inch', 'panel_group', 'refresh_rate_hz', 'resolution_width', 'resolution_height',
    
    # Pin & Sạc
    'battery_mah', 'fast_charge_w',
    
    # Camera chi tiết
    'front_camera_mp', 'rear_main_camera_mp', 'rear_camera_count', 'max_video_resolution', 'fps_at_max_resolution',
    
    # Thiết kế gốc
    'weight_g',           # Cân nặng số thực gốc (ví dụ: 187g)
]

Nhóm 1: Điều kiện cứng (Hard Filters - Bắt buộc thỏa mãn)

- Giá tiền tối đa.

- Phân khúc ưu tiên: Giá rẻ / Flagship cao cấp / Bất kỳ.

Nhóm 2: Mục đích sử dụng (Soft Constraints / Scoring - Dùng để xếp hạng)

- Liên lạc và cơ bản.

- Chơi game.

- Giải trí và Mạng xã hội.

- Chụp hình, quay phim.

- Pin trâu, sạc nhanh.

In [138]:
def calculate_feature_scores(df):
    """
    Hàm tổng hợp các biến vật lý thành 4 cột điểm (thang điểm 10) cho KBRS.
    """
    df_scored = df.copy()

    # Hàm Min-Max Scaling cục bộ (Đưa một cột số về khoảng 0.0 - 1.0)
    def min_max_scale(series):
        return (series - series.min()) / (series.max() - series.min() + 1e-9) # +1e-9 để tránh chia cho 0

    # ---------------------------------------------------------
    # 1. TÍNH ĐIỂM HIỆU NĂNG (score_perf)
    # Giả sử RAM chiếm 60% sức mạnh, Bộ nhớ trong chiếm 40%
    # ---------------------------------------------------------

    # Chuẩn hóa cả 3 tiêu chí về thang [0, 1]
    norm_chip = min_max_scale(df_scored['chip_rating'])
    norm_ram = min_max_scale(df_scored['ram_gb'])
    norm_storage = min_max_scale(df_scored['storage_gb'])

    # Tính điểm với trọng số: Chip (50%), RAM (30%), ROM (20%) - Nhân 10 để ra thang điểm 10
    df_scored['score_perf'] = ((norm_chip * 0.5) + (norm_ram * 0.3) + (norm_storage * 0.2)) * 10

# ---------------------------------------------------------
# 2. TÍNH ĐIỂM CAMERA (score_cam) THEO TRỌNG SỐ MỚI
# ---------------------------------------------------------

    df_scored['score_cam'] = min_max_scale(df_scored['camera_rating_final']) * 10
# ---------------------------------------------------------
# 3. TÍNH ĐIỂM PIN & SẠC (score_batt)
# Pin to chiếm 50%, Sạc nhanh chiếm 50%
# ---------------------------------------------------------
    norm_battery = min_max_scale(df_scored['battery_mah'])
    norm_charge = min_max_scale(df_scored['fast_charge_w'])
    
    df_scored['score_batt'] = ((norm_battery * 0.5) + (norm_charge * 0.5)) * 10

# ---------------------------------------------------------
# 4. TÍNH ĐIỂM MÀN HÌNH (score_disp)
# Tổng số pixel (độ nét) và Tần số quét (độ mượt)
# ---------------------------------------------------------
    norm_dis = min_max_scale(df_scored['display_tier'])
    norm_hz = min_max_scale(df_scored['refresh_rate_hz_fixed'])
    
    df_scored['score_disp'] = ((norm_dis * 0.6) + (norm_hz * 0.4)) * 10

    # Làm tròn điểm về 2 chữ số thập phân cho gọn
    score_cols = ['score_perf', 'score_cam', 'score_batt', 'score_disp']
    df_scored[score_cols] = df_scored[score_cols].round(2)

    return df_scored[final_export_features]

In [139]:
df_ready_for_kbrs = calculate_feature_scores(df_final)

## Xuất file

In [140]:
output_path="../../data/final/scored_data.csv"
df_ready_for_kbrs.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Đã lưu dữ liệu thành công vào: {output_path}")

Đã lưu dữ liệu thành công vào: ../../data/final/scored_data.csv
